In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import re

# ✅ run 폴더
run_dir = Path("../runs/exp1_qwen2p5_baseline_20260317_140322")

print("run_dir exists:", run_dir.exists(), run_dir)
print("results.csv exists:", (run_dir/"results.csv").exists())
print("traces dir exists:", (run_dir/"traces").exists())

run_dir exists: True ../runs/exp1_qwen2p5_baseline_20260317_140322
results.csv exists: False
traces dir exists: True


In [2]:
trace_dir = run_dir / "traces"

all_jsons = sorted(trace_dir.glob("*.json"))
trace_files = [p for p in all_jsons if not p.name.endswith(".edit.json")]

print("all json files  :", len(all_jsons))
print("trace json files:", len(trace_files))
trace_files[:10]

all json files  : 595
trace json files: 300


[PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/astropy__astropy-12907_trial0.json'),
 PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/astropy__astropy-14182_trial0.json'),
 PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/astropy__astropy-14365_trial0.json'),
 PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/astropy__astropy-14995_trial0.json'),
 PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/astropy__astropy-6938_trial0.json'),
 PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/astropy__astropy-7746_trial0.json'),
 PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/django__django-10914_trial0.json'),
 PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/django__django-10924_trial0.json'),
 PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/django__django-11001_trial0.json'),
 PosixPath('../runs/exp1_qwen2p5_baseline_20260317_140322/traces/django__django-1

In [3]:
def pick(*vals):
    for v in vals:
        if v is not None:
            return v
    return None

def nested_get(d, *path):
    cur = d
    for p in path:
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return None
    return cur

def extract_taxonomy_fields(obj, fallback_name=None):
    instance_id = pick(
        obj.get("instance_id"),
        obj.get("task_id"),
        obj.get("problem_id"),
        obj.get("sample_id"),
        fallback_name,
    )

    error_type = pick(
        obj.get("error_type"),
        nested_get(obj, "result", "error_type"),
        nested_get(obj, "analysis", "error_type"),
        nested_get(obj, "failure", "error_type"),
        nested_get(obj, "classification", "error_type"),
    )

    stage = pick(
        obj.get("stage"),
        nested_get(obj, "result", "stage"),
        nested_get(obj, "analysis", "stage"),
        nested_get(obj, "failure", "stage"),
        nested_get(obj, "classification", "stage"),
    )

    signature = pick(
        obj.get("signature"),
        nested_get(obj, "result", "signature"),
        nested_get(obj, "analysis", "signature"),
        nested_get(obj, "failure", "signature"),
        nested_get(obj, "classification", "signature"),
    )

    success = pick(
        obj.get("success"),
        nested_get(obj, "result", "success"),
        nested_get(obj, "analysis", "success"),
        nested_get(obj, "failure", "success"),
        nested_get(obj, "classification", "success"),
    )

    stderr = pick(
        obj.get("stderr"),
        nested_get(obj, "result", "stderr"),
        nested_get(obj, "analysis", "stderr"),
        nested_get(obj, "failure", "stderr"),
    )

    stdout = pick(
        obj.get("stdout"),
        nested_get(obj, "result", "stdout"),
        nested_get(obj, "analysis", "stdout"),
        nested_get(obj, "failure", "stdout"),
    )

    return {
        "instance_id": instance_id,
        "error_type": error_type,
        "stage": stage,
        "signature": signature,
        "success": success,
        "stderr": stderr,
        "stdout": stdout,
    }

In [4]:
rows = []

for p in trace_files:
    try:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)
    except Exception as e:
        rows.append({
            "file": p.name,
            "instance_id": p.stem,
            "error_type": "JSON_READ_ERROR",
            "stage": None,
            "signature": str(e),
            "success": None,
            "stderr": None,
            "stdout": None,
        })
        continue

    row = extract_taxonomy_fields(obj, fallback_name=p.stem)
    row["file"] = p.name
    rows.append(row)

df = pd.DataFrame(rows)

print(df.shape)
display(df.head(10))

(300, 8)


,instance_id,error_type,stage,signature,success,stderr,stdout,file
0,astropy__astropy-12907,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/astropy/modeling/separable.py b/a...,astropy__astropy-12907_trial0.json
1,astropy__astropy-14182,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/astropy/io/ascii/tests/test_rst.p...,astropy__astropy-14182_trial0.json
2,astropy__astropy-14365,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/astropy/io/ascii/tests/test_qdp.p...,astropy__astropy-14365_trial0.json
3,astropy__astropy-14995,APPLY_FAIL,EDIT_APPLY,edit_apply_path_missing,None,astropy/nddata/_core.py,,astropy__astropy-14995_trial0.json
4,astropy__astropy-6938,EDIT_PARSE_FAIL,EDIT_PARSE,invalid_edit_script,None,"Expecting ',' delimiter: line 9 column 10 (cha...",,astropy__astropy-6938_trial0.json
5,astropy__astropy-7746,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/astropy/wcs/wcs.py b/astropy/wcs/...,astropy__astropy-7746_trial0.json
6,django__django-10914,APPLY_FAIL,EDIT_APPLY,edit_apply_path_missing,None,docs/ref/settings.rst,,django__django-10914_trial0.json
7,django__django-10924,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/tests/str/models.py b/tests/str/m...,django__django-10924_trial0.json
8,django__django-11001,APPLY_FAIL,EDIT_APPLY,edit_apply_path_missing,None,django/test/sql/compiler.py,,django__django-11001_trial0.json
9,django__django-11019,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/django/forms/widgets.py b/django/...,django__django-11019_trial0.json


In [5]:
print("error_type counts")
display(df["error_type"].fillna("MISSING").value_counts(dropna=False))

print("\nstage counts")
display(df["stage"].fillna("MISSING").value_counts(dropna=False))

print("\nsignature counts")
display(df["signature"].fillna("MISSING").value_counts(dropna=False).head(30))

print("\nsuccess counts")
display(df["success"].value_counts(dropna=False))

error_type counts


error_type
PRED_READY         212
APPLY_FAIL          72
EDIT_PARSE_FAIL     11
GEN_FAIL             5
Name: count, dtype: int64


stage counts


stage
DIFF_EXPORT    212
EDIT_APPLY      72
EDIT_PARSE      11
GEN              5
Name: count, dtype: int64


signature counts


signature
ready_for_harness          212
edit_apply_path_missing     62
invalid_edit_script         11
edit_apply_range_oob        10
context_length_exceeded      5
Name: count, dtype: int64


success counts


success
None    300
Name: count, dtype: int64

In [6]:
PRE_HARNESS_ERROR_TYPES = [
    "PRED_READY",
    "GEN_FAIL",
    "EDIT_PARSE_FAIL",
    "REPO_FAIL",
    "PATCH_FAIL",
    "APPLY_FAIL",
    "EXEC_EXCEPTION",
    "TIMEOUT",
]

PRE_HARNESS_STAGES = [
    "GEN",
    "REPO",
    "EDIT_PARSE",
    "EDIT_APPLY",
    "DIFF_EXPORT",
]

df_pre = df[
    df["error_type"].isin(PRE_HARNESS_ERROR_TYPES)
    | df["stage"].isin(PRE_HARNESS_STAGES)
].copy()

print(df_pre.shape)
display(df_pre.head(20))

(300, 8)


,instance_id,error_type,stage,signature,success,stderr,stdout,file
0,astropy__astropy-12907,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/astropy/modeling/separable.py b/a...,astropy__astropy-12907_trial0.json
1,astropy__astropy-14182,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/astropy/io/ascii/tests/test_rst.p...,astropy__astropy-14182_trial0.json
2,astropy__astropy-14365,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/astropy/io/ascii/tests/test_qdp.p...,astropy__astropy-14365_trial0.json
3,astropy__astropy-14995,APPLY_FAIL,EDIT_APPLY,edit_apply_path_missing,None,astropy/nddata/_core.py,,astropy__astropy-14995_trial0.json
4,astropy__astropy-6938,EDIT_PARSE_FAIL,EDIT_PARSE,invalid_edit_script,None,"Expecting ',' delimiter: line 9 column 10 (cha...",,astropy__astropy-6938_trial0.json
5,astropy__astropy-7746,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/astropy/wcs/wcs.py b/astropy/wcs/...,astropy__astropy-7746_trial0.json
6,django__django-10914,APPLY_FAIL,EDIT_APPLY,edit_apply_path_missing,None,docs/ref/settings.rst,,django__django-10914_trial0.json
7,django__django-10924,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/tests/str/models.py b/tests/str/m...,django__django-10924_trial0.json
8,django__django-11001,APPLY_FAIL,EDIT_APPLY,edit_apply_path_missing,None,django/test/sql/compiler.py,,django__django-11001_trial0.json
9,django__django-11019,PRED_READY,DIFF_EXPORT,ready_for_harness,None,,diff --git a/django/forms/widgets.py b/django/...,django__django-11019_trial0.json


In [7]:
print("=== stage (where) ===")
display(df_pre["stage"].fillna("MISSING").value_counts())

print("\n=== error_type (what) ===")
display(df_pre["error_type"].fillna("MISSING").value_counts())

print("\n=== signature (why/how) ===")
display(df_pre["signature"].fillna("MISSING").value_counts().head(30))

=== stage (where) ===


stage
DIFF_EXPORT    212
EDIT_APPLY      72
EDIT_PARSE      11
GEN              5
Name: count, dtype: int64


=== error_type (what) ===


error_type
PRED_READY         212
APPLY_FAIL          72
EDIT_PARSE_FAIL     11
GEN_FAIL             5
Name: count, dtype: int64


=== signature (why/how) ===


signature
ready_for_harness          212
edit_apply_path_missing     62
invalid_edit_script         11
edit_apply_range_oob        10
context_length_exceeded      5
Name: count, dtype: int64

In [8]:
df_pre["is_pred_ready"] = df_pre["error_type"].eq("PRED_READY")

print(df_pre["is_pred_ready"].value_counts(dropna=False))

display(
    df_pre.groupby(["stage", "error_type"]).size().sort_values(ascending=False)
)

is_pred_ready
True     212
False     88
Name: count, dtype: int64


stage        error_type     
DIFF_EXPORT  PRED_READY         212
EDIT_APPLY   APPLY_FAIL          72
EDIT_PARSE   EDIT_PARSE_FAIL     11
GEN          GEN_FAIL             5
dtype: int64

In [9]:
sig = df_pre["signature"].dropna().value_counts().index[0]
print("selected signature:", sig)

tmp = df_pre[df_pre["signature"] == sig].head(3)

for _, r in tmp.iterrows():
    print("=" * 100)
    print("instance_id:", r["instance_id"])
    print("stage:", r["stage"])
    print("error_type:", r["error_type"])
    print("signature:", r["signature"])
    print("\n[stderr]")
    print((r["stderr"] or "")[:2000])
    print("\n[stdout]")
    print((r["stdout"] or "")[:2000])

selected signature: ready_for_harness
instance_id: astropy__astropy-12907
stage: DIFF_EXPORT
error_type: PRED_READY
signature: ready_for_harness

[stderr]


[stdout]
diff --git a/astropy/modeling/separable.py b/astropy/modeling/separable.py
index a308e27297..65b76fec6a 100644
--- a/astropy/modeling/separable.py
+++ b/astropy/modeling/separable.py
@@ -98,7 +98,14 @@ def separability_matrix(transform):
         return np.ones((transform.n_outputs, transform.n_inputs),
                        dtype=np.bool_)
     separable_matrix = _separable(transform)
-    separable_matrix = np.where(separable_matrix != 0, True, False)
+def _is_separable_nested(compound_model):
+    return all(isinstance(submodel, CompoundModel) or submodel.n_inputs == 1 and submodel.n_outputs == 1 for submodel in compound_model.models)
+
+def separability_matrix(model):
+    if isinstance(model, CompoundModel) and _is_separable_nested(model):
+        return np.eye(len(model.models), dtype=bool)
+    else:
+        ret

In [11]:
summary = (
    df_pre.groupby(["stage", "error_type", "signature"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

# 비율 추가 (전체 대비)
total = summary["count"].sum()
summary["ratio"] = summary["count"] / total

display(summary)

,stage,error_type,signature,count,ratio
0,DIFF_EXPORT,PRED_READY,ready_for_harness,212,0.706667
1,EDIT_APPLY,APPLY_FAIL,edit_apply_path_missing,62,0.206667
3,EDIT_PARSE,EDIT_PARSE_FAIL,invalid_edit_script,11,0.036667
2,EDIT_APPLY,APPLY_FAIL,edit_apply_range_oob,10,0.033333
4,GEN,GEN_FAIL,context_length_exceeded,5,0.016667


In [12]:
summary_fmt = summary.copy()
summary_fmt["ratio"] = (summary_fmt["ratio"] * 100).round(2)

display(summary_fmt)

,stage,error_type,signature,count,ratio
0,DIFF_EXPORT,PRED_READY,ready_for_harness,212,70.67
1,EDIT_APPLY,APPLY_FAIL,edit_apply_path_missing,62,20.67
3,EDIT_PARSE,EDIT_PARSE_FAIL,invalid_edit_script,11,3.67
2,EDIT_APPLY,APPLY_FAIL,edit_apply_range_oob,10,3.33
4,GEN,GEN_FAIL,context_length_exceeded,5,1.67
